In [1]:
import re
import torch
from transformers import AutoModelForTokenClassification, RobertaTokenizerFast

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
# label_list đúng thứ tự như bạn đã cung cấp (điền lại nếu khác)
label_list = [
    "B-AGE", "B-DATE", "B-GENDER", "B-JOB", "B-LOCATION", "B-NAME", "B-ORGANIZATION", "B-PATIENT_ID",
    "B-SYMPTOM_AND_DISEASE", "B-TRANSPORTATION", "I-AGE", "I-DATE", "I-JOB", "I-LOCATION", "I-NAME",
    "I-ORGANIZATION", "I-PATIENT_ID", "I-SYMPTOM_AND_DISEASE", "I-TRANSPORTATION", "O"
]

model_dir = '/content/drive/MyDrive/Ner_RoBERTa_finetuned/roberta-ner-finetuned'
model = AutoModelForTokenClassification.from_pretrained(model_dir)
tokenizer = RobertaTokenizerFast.from_pretrained(model_dir)
id2label = {i: label for i, label in enumerate(label_list)}


In [9]:
def split_sentences(text):
    sentences = re.split(r'(?<=[.!?…])\s+', text.strip())
    sentences = [s for s in sentences if s.strip()]
    return sentences

def merge_roberta_subwords(tokens, labels):
    words = []
    word_labels = []
    current_word = ""
    current_label = None
    for token, label in zip(tokens, labels):
        if token in ["<s>", "</s>", "<pad>"]:
            continue
        if token.startswith("Ġ"):
            if current_word != "":
                words.append(current_word)
                word_labels.append(current_label if current_label is not None else label)
            current_word = token[1:]
            current_label = label
        else:
            current_word += token
    if current_word != "":
        words.append(current_word)
        word_labels.append(current_label if current_label is not None else label)
    return words, word_labels

def predict_entities_for_text(text):
    sentences = split_sentences(text)
    print(f"Phát hiện {len(sentences)} câu trong đoạn văn.")
    for idx, sent in enumerate(sentences, 1):
        print(f"\n------ Câu {idx} ------")
        # Tách từ (dùng .split() để lấy danh sách từ)
        words = sent.split()
        # Tokenize với is_split_into_words=True để mapping từng từ
        encoded = tokenizer(words, return_tensors="pt", padding=True, truncation=True, max_length=128, is_split_into_words=True)
        with torch.no_grad():
            outputs = model(input_ids=encoded["input_ids"], attention_mask=encoded["attention_mask"])
            logits = outputs.logits

        predictions = torch.argmax(logits, dim=-1)[0].tolist()
        word_ids = encoded.word_ids(batch_index=0)

        # Mapping label cho từng từ
        final_words = []
        final_labels = []
        previous_word_idx = None
        for idx, word_idx in enumerate(word_ids):
            if word_idx is None:
                continue
            # Chỉ lấy label tại token đầu tiên của từ (hoặc bạn muốn vote thì tùy)
            if word_idx != previous_word_idx:
                final_words.append(words[word_idx])
                final_labels.append(id2label[predictions[idx]])
                previous_word_idx = word_idx

        print(f"{'Word':<20} {'Predicted Label'}")
        print("-" * 40)
        for word, label in zip(final_words, final_labels):
            print(f"{word:<20} {label}")


# Dùng thử
text = input("Nhập đoạn văn để nhận dạng thực thể: ")
predict_entities_for_text(text)

Nhập đoạn văn để nhận dạng thực thể: Nhập đoạn văn để nhận dạng thực thể: Hôm nay, ngày 8 tháng 6 năm 2025, Nguyễn Văn An, một kỹ sư phần mềm, đã tham gia hội thảo công nghệ tại Hà Nội. Anh ấy làm việc cho công ty Công nghệ FPT, một trong những tập đoàn lớn nhất Việt Nam. Hội thảo được tổ chức tại Trung tâm Hội nghị Quốc gia, nơi thu hút hơn 500 chuyên gia từ khắp nơi. An đã gặp Trần Thị Bình, một nhà nghiên cứu AI đến từ Đại học Bách Khoa. Họ cùng thảo luận về dự án trí tuệ nhân tạo với Google
Phát hiện 5 câu trong đoạn văn.

------ Câu 1 ------
Word                 Predicted Label
----------------------------------------
Nhập                 O
đoạn                 O
văn                  O
để                   O
nhận                 O
dạng                 O
thực                 O
thể:                 O
Hôm                  O
nay,                 O
ngày                 O
8                    B-DATE
tháng                B-DATE
6                    I-DATE
năm                  O
2025,    